# W10 -- Poster Figures

Interactive Plotly figures used in the Week 10 poster draft:

1. Price-to-income ratio over time for the three training cities, with the 5.0 collapse threshold marked.
2. Top 20 holdout metros by predicted affordability-risk score (Model 2).
3. Top 5 holdout metros, styled for the poster layout.

**Inputs:** `data/final_data/price_changes_with_collapse_flags.csv` and `notebooks/w07-data-summary/output/tables/holdout_city_risk_scores.csv` (produced by `w07-data-summary.ipynb`).

In [ ]:
import pandas as pd
import plotly.express as px

DATA_PATH = "../../data/final_data/price_changes_with_collapse_flags.csv"
RISK_SCORES_PATH = "../w07-data-summary/output/tables/holdout_city_risk_scores.csv" 

## Figure 1: Price-to-income ratio, training cities

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)

metros = ["Austin-Round Rock-San Marcos, TX", "Boise City, ID", "Tampa, FL (MSAD)"]
sub = df[df["metro_name.x"].isin(metros)].copy()

sub["date"] = pd.to_datetime(
    sub["year"].astype(int).astype(str) + "-" + ((sub["qtr"].astype(int) - 1) * 3 + 1).astype(str) + "-01"
)
sub["metro"] = sub["metro_name.x"].map({metros[0]: "Austin", metros[1]: "Boise", metros[2]: "Tampa"})
sub = sub.dropna(subset=["price_to_income_ratio"])

fig = px.line(
    sub, x="date", y="price_to_income_ratio", color="metro",
    title="Price-to-Income Ratio: Austin, Boise, Tampa"
)
fig.add_hline(y=5.0, line_dash="dash", line_color="black")
fig.show()

## Figure 2: Top 20 holdout metros by predicted risk

In [ ]:
risk_df = pd.read_csv(RISK_SCORES_PATH)
top20 = risk_df.sort_values("risk_score", ascending=False).head(20)

fig = px.bar(
    top20, x="risk_score", y="metro_name_x", orientation="h",
    title="Top 20 Holdout Metros by Predicted Affordability Risk Score",
    labels={"risk_score": "Predicted Risk Score (Probability of Unaffordability)", "metro_name_x": "Metro"},
    color="risk_score", color_continuous_scale="Reds"
)
fig.update_layout(
    yaxis=dict(autorange="reversed"),
    template="plotly_white",
    width=1000, height=700
)
fig.show()

## Figure 3: Top 5 holdout metros (poster-styled)

Same data as Figure 2, narrowed to the top 5 and styled for the poster: larger fonts, a subtitle, and value labels on each bar.

In [ ]:
risk_df = pd.read_csv(RISK_SCORES_PATH)
top5 = risk_df.sort_values("risk_score", ascending=False).head(5)

fig = px.bar(
    top5,
    x="risk_score",
    y="metro_name_x",
    orientation="h",
    labels={
        "risk_score": "Predicted Risk Score (Probability of Unaffordability)",
        "metro_name_x": "Metro"
    },
    color="risk_score",
    color_continuous_scale="Reds",
    text="risk_score"  # displays the score at the end of each bar
)

# Format displayed scores and allow labels outside the bars
fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside",
    cliponaxis=False
)

# Customize titles, font sizes, axes, and chart spacing
fig.update_layout(
    title={
        "text": "Top 5 Holdout Metros by Predicted Affordability Risk Score",
        "subtitle": {
            "text": "Higher scores indicate stronger resemblance to pre-collapse Tampa, Austin, and Boise patterns",
            "font": {"size": 20, "color": "gray"}
        },
        "x": 0.5,
        "xanchor": "center",
        "font": {"size": 32}
    },
    font={"family": "Arial", "size": 22},
    xaxis={
        "title_font": {"size": 25},
        "tickfont": {"size": 20},
        "range": [0, 0.72],  # zoom the axis to make score differences visible
        "showgrid": True,
        "gridcolor": "#E5E5E5"
    },
    yaxis={
        "title_font": {"size": 25},
        "tickfont": {"size": 22},
        "autorange": "reversed"  # place the highest-risk metro at the top
    },
    coloraxis_colorbar={
        "title": "Risk Score",
        "title_font": {"size": 19},
        "tickfont": {"size": 17},
        "x": 1.03,
        "len": 0.78
    },
    template="plotly_white",
    width=1400,
    height=800,
    margin={"l": 225, "r": 185, "t": 155, "b": 110}
)
fig.show()